<a href="https://colab.research.google.com/github/MANI-WEBDEVE/Learn_AI/blob/main/machine_learning/Boosting/Ada_Boost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from sklearn import datasets
from sklearn.metrics import accuracy_score
import time
data  = datasets.make_classification(5000000, 10, random_state=42)

In [2]:
X,Y =data


In [3]:
X.shape

(5000000, 10)

In [4]:
Y.shape

(5000000,)

In [5]:
for y in range(len(Y)):
  if Y[y] == 0:
    Y[y] = -1
  elif Y[y] == 1:
    Y[y] = +1

In [6]:
Y

array([-1, -1,  1, ...,  1,  1,  1])

In [17]:
def Ada_Boost_Decision_Stump(X,Y,feature_index,threshold,weight):
  predictions = np.where(X[:,feature_index] > threshold,1,-1)
  error=np.sum(weight * (predictions != Y))
  return predictions,error

In [8]:
print(Ada_Boost_Decision_Stump(X,Y,1,-0.90,0.25))

(array([1, 1, 1, ..., 1, 1, 1]), np.float64(624659.25))


In [23]:
def find_best_stump(X,Y,weight):
  best_stump = {}
  min_error = float('inf')
  n_samples, n_features = X.shape

  for feature_index in range(n_features):
    # print(time.gmtime().tm_min, time.gmtime().tm_sec)
    # thresholds=np.unique(X[:,feature_index])
    # for threshold in thresholds:
    #   predictions, error = Ada_Boost_Decision_Stump(X,Y,feature_index,threshold,weight)
      # print(time.gmtime().tm_min, time.gmtime().tm_sec)
      # threshold = np.mean(X[:,feature_index])
      # predictions, error = Ada_Boost_Decision_Stump(X,Y,feature_index,threshold,weight)
      values = np.unique(X[:, feature_index])
      k = 10
      thresholds = np.linspace(values.min(), values.max(), k)

      for threshold in thresholds:
        predictions, error = Ada_Boost_Decision_Stump(X,Y,feature_index,threshold,weight)

        if error < min_error:
          min_error = error
          best_stump = {
          'feature_idx': feature_index,
          'threshold': threshold,
          'predictions': predictions,
          'error': error
      }

  return best_stump , error

In [10]:
def ada_boost(X,Y, estimator=5):
  N = len(Y)
  weights = np.full(N, 1/N)
  stumps=[]
  alphas=[]

  for i in range(estimator):
    best_stump,error = find_best_stump(X,Y,weights)
    predictions = best_stump['predictions']
    error = best_stump['error']
    feature_idx = best_stump['feature_idx']
    threshold = best_stump['threshold']

    # print(f" Selected stump: {'feature_names'[feature_idx]} > {threshold}")
    # print(f" Predictions: {predictions}")
    # print(f" Actual y:    {y}")
    # print(f" Weighted error = {error:.6f}")


    if error == 0:
      alpha = 10.0 # special case
    else:
      alpha = 0.5 * np.log((1 - error) / error)

    alphas.append(alpha)

    # weights update
    update_factor = np.exp(-alpha * Y * predictions) # ex = 0.67
    new_weights = weights * update_factor # ex = 0.25 x 0.67 =
    new_weights = new_weights / np.sum(new_weights) # [0.1675, 0.1675, 0.1666, 0.1675] / 0.6685

    stumps.append(best_stump)
    weights = new_weights


  return stumps,alphas


In [11]:

def predict(X, stumps, alphas):
    final_scores = np.zeros(len(X))
    print(final_scores)
    for t in range(len(stumps)):
        print(t)
        stump = stumps[t]
        pred = np.where(X[:, stump['feature_idx']] > stump['threshold'], +1, -1)
        final_scores += alphas[t] * pred
    #     print(f" Stump {t+1} ({'feature_names'[stump['feature_idx']]}>{stump['threshold']}), alpha={alphas[t]} => pred: {pred}")
    # print(" Final scores per sample (sum alpha*pred):", final_scores)
    return np.sign(final_scores)

In [24]:
stumps, alphas = ada_boost(X,Y, estimator=5)
print("\n" + "="*40)
# print("Predicting on training data:")
y_pred = predict(X, stumps, alphas)
# print("Final predicted labels:", y_pred)
# print("Actual labels:", y)
# print(f"Accuracy: {np.mean(y_pred == y)*100:.1f}%")


[0. 0. 0. ... 0. 0. 0.]
0
1
2
3
4


In [25]:
Y

array([-1, -1,  1, ...,  1,  1,  1])

In [26]:
accuracy_score(Y, y_pred)

0.5000212

In [33]:
import numpy as np
np.linspace(np.array([3,4,2,14,43]).min(),np.array([100,787,32,123]).max(), 10)

array([  2.        ,  89.22222222, 176.44444444, 263.66666667,
       350.88888889, 438.11111111, 525.33333333, 612.55555556,
       699.77777778, 787.        ])